# Performance profiling of distance functions

Anton Antonov    
Maty 2026

---

## Setup

In [ ]:
use Math::DistanceFunctions;

use NativeCall;
use NativeHelpers::Array;

use Data::Summarizers;
use Data::Reshapers;

---

## Verify NativeCall implementation performance

The code in this section is from the example file ["distance-functions-profiling.raku"](https://github.com/antononcube/Raku-Math-DistanceFunctions-Native/blob/main/examples/distance-functions-profiling.raku) of ["Math::DistanceFunctions::Native"](https://github.com/antononcube/Raku-Math-DistanceFunctions-Native).

In [2]:
my $iterations = 1_000;

my @large-vec1 = (^1000).map({1.rand});
my @large-vec2 = (^1000).map({1.rand});
my $large-vec1c = copy-to-carray(@large-vec1, num64);
my $large-vec2c = copy-to-carray(@large-vec2, num64);

NativeCall::Types::CArray[num64].new

In [3]:
say $large-vec1c.WHAT;
say $large-vec1c.elems;
say $large-vec1c ~~ Positional;
say $large-vec1c ~~ Iterable:D;
say $large-vec1c[12]

(CArray[num64])
1000
True
False
0.06645154328795477


In [4]:
my @dist-functions = &cosine-distance, &euclidean-distance, &squared-euclidean-distance, &dot-product;

[&cosine-distance &euclidean-distance &squared-euclidean-distance &dot-product]

In [5]:
my @dsTimings = do for @dist-functions -> &func {
    my $start = now;
    for ^$iterations {
        cosine-distance(@large-vec1, @large-vec2);
    }
    my $time = now - $start;
    {func => &func.name, total-time => $time, mean-time => $time / $iterations, type => 'Array'}
}

my @dsTimingsC = do for @dist-functions -> &func {
    my $start = now;
    for ^$iterations {
        cosine-distance($large-vec1c, $large-vec2c);
    }
    my $time = now - $start;
    {func => &func.name, total-time => $time, mean-time => $time / $iterations, type => 'CArray'}
}

@dsTimings .= append(@dsTimingsC);

deduce-type(@dsTimings);

Vector(Struct([func, mean-time, total-time, type], [Str, Num, Duration, Str]), 8)

Tabulate:

In [6]:
#% html

my @field-names = <type func total-time mean-time>;
@dsTimings
==> {.sort(*<func>)}()
==> to-html(:@field-names)


type,func,total-time,mean-time
Array,cosine-distance,1.187573681,0.001187573681
CArray,cosine-distance,0.006720737,6.720737e-06
Array,dot-product,1.175906173,0.001175906173
CArray,dot-product,0.004166101,4.166101e-06
Array,euclidean-distance,1.176697384,0.001176697384
CArray,euclidean-distance,0.004152935,4.1529349999999996e-06
Array,squared-euclidean-distance,1.17673151,0.00117673151
CArray,squared-euclidean-distance,0.004738771,4.738771e-06


In [7]:
sink records-summary(@dsTimings, :@field-names)

+-------------+---------------------------------+------------------------+----------------------------------+
| type        | func                            | total-time             | mean-time                        |
+-------------+---------------------------------+------------------------+----------------------------------+
| Array  => 4 | squared-euclidean-distance => 2 | Min    => 0.004152935  | Min    => 4.1529349999999996e-06 |
| CArray => 4 | dot-product                => 2 | 1st-Qu => 0.004452436  | 1st-Qu => 4.452436e-06           |
|             | euclidean-distance         => 2 | Mean   => 0.5920859115 | Mean   => 0.0005920859114999999  |
|             | cosine-distance            => 2 | Median => 0.591313455  | Median => 0.000591313455         |
|             |                                 | 3rd-Qu => 1.176714447  | 3rd-Qu => 0.001176714447         |
|             |                                 | Max    => 1.187573681  | Max    => 0.001187573681         |
+---------

In [8]:
sink records-summary(group-by(@dsTimings, <type>), :@field-names)

summary of CArray =>
+-------------+---------------------------------+-----------------------+----------------------------------+
| type        | func                            | total-time            | mean-time                        |
+-------------+---------------------------------+-----------------------+----------------------------------+
| CArray => 4 | dot-product                => 1 | Min    => 0.004152935 | Min    => 4.1529349999999996e-06 |
|             | cosine-distance            => 1 | 1st-Qu => 0.004159518 | 1st-Qu => 4.159518e-06           |
|             | squared-euclidean-distance => 1 | Mean   => 0.004944636 | Mean   => 4.944636e-06           |
|             | euclidean-distance         => 1 | Median => 0.004452436 | Median => 4.452436e-06           |
|             |                                 | 3rd-Qu => 0.005729754 | 3rd-Qu => 5.729754e-06           |
|             |                                 | Max    => 0.006720737 | Max    => 6.720737e-06           